# Voila App Components -- Demonstration Notebook

This notebook demonstrates **reusable code snippets** for building interactive Voila applications. Each section shows you how to use pre-built Python modules that you can copy and adapt for your own projects.

---

## What You'll Learn

This notebook contains four practical examples:

1. **Authentication** -- How to connect to NOMAD using this repo's shared `hysprint_utils.access_token` helper
2. **Tab Creation** -- How to organize your app into multiple tabs using `ipywidgets`
3. **Batch Selection** -- How to create interactive dropdowns that fetch and display NOMAD batch data, using `hysprint_utils.batch_selection`
4. **Resizable Plots** -- How to create professional, resizable Plotly charts

Also remember:
1. If you find an application that you want to adapt, make sure to copy and rename the containing folder and adapt it to your own needs.
2. Don't be afraid of using the chatbot you trust the most.

**💡 Each section is independent** -- you can run them in any order and copy the code for your own applications.


## 0. Running code cells

- Run a cell with **Shift + Enter**
- Run **Setup** first, then **Section 1 (Authentication)** before Section 3 -- the batch selector needs a token.


## Setup: connect this notebook to `hysprint_utils`

The authentication and batch-selection helpers used below (`access_token.py`, `batch_selection.py`, `api_calls.py`) all live in [`shared/hysprint_utils/`](../shared/hysprint_utils/) -- the same package every app in this repo imports from, so what you learn here is exactly what powers the real apps. The resizable-plot helper in Section 4 is a bit different -- see the note there.

**What these libraries do:**
- `ipywidgets` -- Creates interactive buttons, dropdowns, and tabs in Jupyter notebooks
- `plotly` -- Creates interactive, publication-quality charts and graphs
- `pandas` / `numpy` -- data handling (needed by some modules)


In [ ]:
import subprocess
import sys
from pathlib import Path

# Learning/ sits next to shared/ at the repo root
_shared = (Path.cwd() / "../shared").resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(_shared)], check=True)
if str(_shared) not in sys.path:
    sys.path.insert(0, str(_shared))
del _shared

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display  # noqa: F401
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
pio.renderers.default = "notebook"

print("✅ Base libraries imported successfully")


---

# Section 1: Authentication with NOMAD

**What this section does:**
Connects to NOMAD using `hysprint_utils.access_token.get_token`, then makes one authenticated request to see how the token is actually used.

**How it works:**
1. `get_token(url)` finds your token (environment variable, local `secrets.py`, or an interactive prompt as a last resort) -- see `03_HandlingJVdata.ipynb` for the full explanation of that fallback chain.
2. We use it to call `GET /users/me`, NOMAD's "who am I" endpoint, to confirm it works and see our own user info.

**📖 What you can learn from this code:**
- How to attach a bearer token to an authenticated `requests` call
- How to handle authentication errors gracefully
- **Why you should never print a full token** (see the callout in the cell below)


In [ ]:
import requests

from hysprint_utils.access_token import get_token
from hysprint_utils.config import API_ENDPOINT, URL_BASE

NOMAD_URL = f"{URL_BASE}{API_ENDPOINT}"

print("=" * 70)
print("NOMAD AUTHENTICATION")
print("=" * 70)

AUTHENTICATED_TOKEN = None
AUTHENTICATED_USER = None

try:
    AUTHENTICATED_TOKEN = get_token(NOMAD_URL)

    # This is the part worth understanding: any authenticated NOMAD API call
    # attaches the token the same way, via an 'Authorization: Bearer <token>' header.
    verify_url = f"{NOMAD_URL}/users/me"
    headers = {"Authorization": f"Bearer {AUTHENTICATED_TOKEN}"}
    response = requests.get(verify_url, headers=headers, timeout=10)
    response.raise_for_status()

    user_info = response.json()
    AUTHENTICATED_USER = user_info.get("name", user_info.get("username", "Unknown"))

    print(f"✅ Logged in as: {AUTHENTICATED_USER}")
    print(f"📧 Email: {user_info.get('email', 'N/A')}")
    # ⚠️ Never print a full token -- if this cell's output is saved (it is, the
    # moment you run it) and the notebook is later shared or committed, that output
    # is a live credential. Always slice it, even "just for debugging":
    print(f"🔑 Token preview: {AUTHENTICATED_TOKEN[:20]}...")
    print("\n✅ Ready to proceed with data loading!")

except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("\n⚠️  Please ensure you are logged into NOMAD and have network connectivity.")


---

# Section 2: Creating Tabbed Interfaces

**What this section does:**
Shows you how to organize your application into multiple tabs, similar to how the JV Analysis app is structured.

**Why use tabs:**
- Keeps your interface organized and uncluttered
- Groups related functionality together
- Makes complex applications easier to navigate

**📖 Key concepts to learn:**
- `widgets.Tab()` -- The main container for tabbed content
- `widgets.VBox()` -- Stacks widgets vertically
- `widgets.Output()` -- Creates an area where you can print messages
- `.on_click()` -- Attaches a function to run when a button is clicked

**💡 Try this:** After running this cell, click the buttons in different tabs to see how they respond!


In [ ]:
# Create output areas for each tab
tab1_output = widgets.Output()
tab2_output = widgets.Output()
tab3_output = widgets.Output()

# Create buttons
auth_button = widgets.Button(description="Authenticate", button_style="primary")
load_button = widgets.Button(description="Load Data", button_style="success")
plot_button = widgets.Button(description="Generate Plot", button_style="info")


# Button click handlers
def on_auth_click(b):
    with tab1_output:
        clear_output()
        print("🔘 Authenticate button pressed!")


def on_load_click(b):
    with tab2_output:
        clear_output()
        print("🔘 Load Data button pressed!")


def on_plot_click(b):
    with tab3_output:
        clear_output()
        print("🔘 Generate Plot button pressed!")


# Attach handlers
auth_button.on_click(on_auth_click)
load_button.on_click(on_load_click)
plot_button.on_click(on_plot_click)

# Create content for each tab
tab1_content = widgets.VBox([
    widgets.HTML("<h3>Connection Settings</h3>"),
    widgets.Label("This tab would contain authentication widgets, it is just a dummy example"),
    widgets.Text(placeholder="Enter username", description="Username:"),
    widgets.Password(placeholder="Enter password", description="Password:"),
    auth_button,
    tab1_output,
])

tab2_content = widgets.VBox([
    widgets.HTML("<h3>Batch Selection</h3>"),
    widgets.Label("This tab would contain batch selection widgets"),
    widgets.SelectMultiple(
        options=["Batch_001", "Batch_002", "Batch_003"],
        description="Batches:",
        layout=widgets.Layout(width="400px", height="200px"),
    ),
    load_button,
    tab2_output,
])

tab3_content = widgets.VBox([
    widgets.HTML("<h3>Data Analysis</h3>"),
    widgets.Label("This tab would contain analysis tools and plots"),
    widgets.Dropdown(
        options=["Boxplot", "Histogram", "JV Curve"],
        description="Plot Type:",
        value="Boxplot",
    ),
    plot_button,
    tab3_output,
])

# Create the tab widget
tabs = widgets.Tab(children=[tab1_content, tab2_content, tab3_content])
tabs.set_title(0, "Connection")
tabs.set_title(1, "Batches")
tabs.set_title(2, "Analysis")

# Display the tabs
display(tabs)
print("✅ Tabbed interface created successfully")
print("💡 Click on different tabs to switch between views")
print("💡 Try clicking the buttons in each tab!")


---

# Section 3: Batch Selection Widget

**What this section does:**
Creates an interactive dropdown that fetches batch names from NOMAD and lets users select which batches to load data from.

**How it works:**
1. Calls `hysprint_utils.api_calls.get_batch_ids` to get all available batch IDs
2. `hysprint_utils.batch_selection.create_batch_selection` builds a searchable multi-select widget showing all batches
3. When "Load Data" is clicked, prints which batches were selected

**Prerequisites:**
- ⚠️ You must run Section 1 (Authentication) first to get your NOMAD token

**📖 What you can learn from this code:**
- How to fetch data from an API and display it in a widget
- How to use `widgets.SelectMultiple` for multi-selection
- How to implement search/filter functionality
- How to pass callback functions to handle button clicks

**💡 In a real app:** Instead of printing selected batches, you would load the actual sample data and measurements for analysis -- exactly like `03_HandlingJVdata.ipynb` does.


In [ ]:
# Import required modules
try:
    from hysprint_utils.api_calls import get_batch_ids  # noqa: F401
    from hysprint_utils.batch_selection import create_batch_selection

    print("✅ Batch selection modules imported successfully")
except ImportError as e:
    print(f"❌ Error importing batch selection modules: {e}")
    print("Make sure the Setup cell above ran successfully")

# Check if authenticated
token = AUTHENTICATED_TOKEN
if token:
    print("✅ Using authenticated token")
else:
    print("⚠️  No authentication token found. Please run Section 1 first.")

# Output area for batch loading
batch_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd", padding="10px", margin="10px 0", min_height="100px"
    )
)


def load_batch_data(batch_selector_widget):
    """Handler function called when Load Data button is clicked"""
    with batch_output:
        clear_output(wait=True)
        selected_batches = batch_selector_widget.value

        if not selected_batches:
            print("⚠️  No batches selected")
            return

        print("✅ Load Data button clicked!")
        print(f"\n📦 Selected Batches ({len(selected_batches)}):")
        for i, batch in enumerate(selected_batches, 1):
            print(f"  {i}. {batch}")

        print("\n💾 Next steps would be:")
        print("  - Query NOMAD for samples in these batches (get_ids_in_batch)")
        print("  - Load JV measurement data (get_all_JV)")
        print("  - Process and display the data")


if token:
    try:
        print("\n🔄 Fetching batch IDs from NOMAD...")
        batch_widget = create_batch_selection(NOMAD_URL, token, load_batch_data)
        print("✅ Batch selection widget created successfully")

        display(widgets.VBox([
            widgets.HTML("<h3>Batch Selection</h3>"),
            widgets.HTML(
                "<p>Search for batches and select multiple items using "
                "Ctrl/Cmd + Click</p>"
            ),
            batch_widget,
            batch_output,
        ]))

    except Exception as e:
        print(f"❌ Error creating batch selection widget: {e}")
else:
    print("\n⚠️  Cannot create batch selection widget without authentication")


---

# Section 4: Creating Resizable Interactive Plots

**What this section does:**
Shows you how to create professional Plotly charts that users can resize by dragging the corner, making your visualizations more flexible and user-friendly.

**ℹ️ Note on where this helper lives:** unlike Sections 1 and 3, `resizable_plot_utility.py` is not (yet) part of the shared `hysprint_utils` package -- it currently lives in [`apps/JV-Analysis/`](../apps/JV-Analysis/resizable_plot_utility.py), where it was first built. The cell below reaches it with a small extra `sys.path` entry, just for this demo. If you find yourself wanting this in more than one app, the right move is to propose promoting it into `hysprint_utils` (see `CLAUDE.md`) rather than copy-pasting the file around.

**📖 Key function to use in your code:**
```python
display_resizable_plot(fig, title, width, height)
```
- `fig` -- Your Plotly figure object
- `title` -- Title to display above the plot
- `width` / `height` -- Initial size in pixels

**💡 After running this cell:** Try dragging the bottom-right corner of each plot to resize it.


In [ ]:
import sys
from pathlib import Path

# resizable_plot_utility.py is app-local to JV-Analysis (see note above),
# not part of hysprint_utils -- so it needs its own sys.path entry.
_jv_analysis_dir = (Path.cwd() / "../apps/JV-Analysis").resolve()
if str(_jv_analysis_dir) not in sys.path:
    sys.path.insert(0, str(_jv_analysis_dir))

from resizable_plot_utility import display_resizable_plot

# Example 1: Line Plot
print("Creating Example 1: Trigonometric Functions...")
x = np.linspace(0, 10, 100)
y1 = np.sin(x) + np.random.normal(0, 0.1, 100)
y2 = np.cos(x) + np.random.normal(0, 0.1, 100)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=x, y=y1, mode="lines+markers", name="Sin Wave",
                          line=dict(color="#1f77b4", width=2), marker=dict(size=4)))
fig1.add_trace(go.Scatter(x=x, y=y2, mode="lines+markers", name="Cos Wave",
                          line=dict(color="#ff7f0e", width=2), marker=dict(size=4)))
fig1.update_layout(title="Trigonometric Functions with Noise", xaxis_title="X Values",
                   yaxis_title="Y Values", template="plotly_white", hovermode="x unified")

display_resizable_plot(fig1, "Example 1: Trigonometric Functions", 800, 500)
print("✅ Plot 1 created!\n")

# Example 2: Bar Chart
print("Creating Example 2: Efficiency Comparison...")
batches = ["Batch_A", "Batch_B", "Batch_C", "Batch_D", "Batch_E"]
efficiency_values = [12.5, 14.2, 13.8, 15.1, 13.3]

fig2 = go.Figure()
fig2.add_trace(go.Bar(x=batches, y=efficiency_values,
                      marker=dict(color=efficiency_values, colorscale="Viridis"),
                      text=[f"{val:.1f}%" for val in efficiency_values], textposition="outside"))
fig2.update_layout(title="Power Conversion Efficiency by Batch", xaxis_title="Batch ID",
                   yaxis_title="Parameter (%)", template="plotly_white")

display_resizable_plot(fig2, "Example 2: Efficiency Comparison", 700, 500)
print("✅ Plot 2 created!\n")

print("\n✅ All example plots created successfully!")
print("💡 Drag the bottom-right corner of each plot to resize it")


---

## Summary & Next Steps

**🎉 Congratulations!** You've learned the four essential building blocks for creating Voila applications:

### What You've Learned:

1. **✅ Authentication** -- Connect securely to NOMAD with `hysprint_utils.access_token.get_token`
2. **✅ Tab Creation** -- Organize complex UIs into manageable sections with `widgets.Tab()` and `.on_click()`
3. **✅ Batch Selection** -- Fetch and display NOMAD batches with `hysprint_utils.batch_selection.create_batch_selection`
4. **✅ Resizable Plots** -- Build flexible visualizations with `display_resizable_plot(fig, title, width, height)`
5. **✅ Use LLMs** -- Chatbots can speed up how quickly you develop these apps

### Modules used here, and where they live:
- `hysprint_utils.access_token`, `hysprint_utils.batch_selection`, `hysprint_utils.api_calls` -- shared, already installed for every app via the Setup cell, no copying needed.
- `resizable_plot_utility.py` -- currently app-local to `JV-Analysis` (see the note in Section 4).

### Try it yourself:
- Swap Section 4's example data for one of your own batches (combine it with Section 3's selector).
- Add a third tab-driven example to Section 2 that calls `get_batch_ids` when clicked.
- Open [`api_calls.py`](../shared/hysprint_utils/api_calls.py) and find a function not used in this notebook or in `03_HandlingJVdata.ipynb` -- try it.

### How to Build Your Own App:

1. **Start with a notebook** -- Copy sections from this demo
2. **Add your data processing** -- Load and analyze your specific data
3. **Create your visualizations** -- Use Plotly to create charts
4. **Add interactivity** -- Connect widgets to your analysis functions
5. **Deploy with Voila** -- Run `voila your_notebook.ipynb` to create a web app

### Need Help?
- Check the Plotly documentation: https://plotly.com/python/
- ipywidgets guide: https://ipywidgets.readthedocs.io/
- Ask your colleagues who have built similar apps!

---

For questions or comments contact edgar.nandayapa@helmholtz-berlin.de

Helmholtz-Zentrum Berlin
